In [1]:
"""
Text/NLP Feature Extraction (Simplified)
=========================================
从Reddit文本提取NLP特征

输入: reddit_wsb_for_network.csv
输出: text_features_5min.parquet

核心特征:
1. Sentiment (情感分析)
   - sentiment_positive, sentiment_negative, sentiment_neutral, sentiment_compound
   
2. Urgency (紧迫性指标)
   - urgency_score (基于关键词)
   
3. Financial Keywords (金融术语)
   - bullish_keywords, bearish_keywords
   
4. Emotional Expression (情绪表达)
   - emoji_density, caps_ratio, exclamation_ratio

Requirements:
    pip install vaderSentiment
    
    如果没有安装VADER，会使用简化版sentiment计算
"""

import pandas as pd
import numpy as np
from datetime import datetime
import re
from collections import Counter

# 尝试导入VADER
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    VADER_AVAILABLE = True
except ImportError:
    print("⚠️  VADER not installed. Will use simplified sentiment.")
    print("   To install: pip install vaderSentiment")
    VADER_AVAILABLE = False

# ============================================================
# 参数设置
# ============================================================

TIME_WINDOW = 5  # 分钟
DATA_FILE = 'reddit_wsb_for_network.csv'

# 关键词列表
URGENCY_KEYWORDS = [
    'now', 'urgent', 'hurry', 'asap', 'quick', 'fast', 
    'immediately', 'right now', 'dont wait', "don't wait",
    'last chance', 'limited time', 'act now'
]

BULLISH_KEYWORDS = [
    'moon', 'rocket', 'bull', 'buy', 'long', 'calls',
    'squeeze', 'rally', 'breakout', 'to the moon',
    'diamond hands', 'hold', 'hodl', 'tendies'
]

BEARISH_KEYWORDS = [
    'bear', 'crash', 'dump', 'sell', 'short', 'puts',
    'drop', 'fall', 'tank', 'plummet', 'paper hands'
]

# Emoji正则
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map symbols
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "]+", 
    flags=re.UNICODE
)

# ============================================================
# 核心函数
# ============================================================

def calculate_sentiment_vader(texts):
    """
    使用VADER计算情感分数
    """
    analyzer = SentimentIntensityAnalyzer()
    
    sentiments = []
    for text in texts:
        if pd.isna(text) or str(text).strip() == '':
            sentiments.append({
                'positive': 0,
                'negative': 0,
                'neutral': 1,
                'compound': 0
            })
        else:
            scores = analyzer.polarity_scores(str(text))
            sentiments.append({
                'positive': scores['pos'],
                'negative': scores['neg'],
                'neutral': scores['neu'],
                'compound': scores['compound']
            })
    
    return pd.DataFrame(sentiments)


def calculate_sentiment_simple(texts):
    """
    简化版情感分析（无需VADER）
    基于正面/负面词汇计数
    """
    positive_words = ['good', 'great', 'awesome', 'love', 'best', 'win', 'moon', 'rocket']
    negative_words = ['bad', 'worst', 'hate', 'lose', 'crash', 'dump', 'shit', 'fuck']
    
    sentiments = []
    for text in texts:
        if pd.isna(text) or str(text).strip() == '':
            sentiments.append({
                'positive': 0,
                'negative': 0,
                'neutral': 1,
                'compound': 0
            })
            continue
        
        text_lower = str(text).lower()
        
        pos_count = sum(text_lower.count(word) for word in positive_words)
        neg_count = sum(text_lower.count(word) for word in negative_words)
        
        total = pos_count + neg_count + 1
        
        sentiments.append({
            'positive': pos_count / total,
            'negative': neg_count / total,
            'neutral': 1 - (pos_count + neg_count) / total,
            'compound': (pos_count - neg_count) / total
        })
    
    return pd.DataFrame(sentiments)


def calculate_urgency_score(text):
    """
    计算紧迫性分数
    """
    if pd.isna(text) or str(text).strip() == '':
        return 0
    
    text_lower = str(text).lower()
    
    # 计算紧迫性关键词出现次数
    urgency_count = sum(text_lower.count(keyword) for keyword in URGENCY_KEYWORDS)
    
    # 归一化（相对于文本长度）
    words = text_lower.split()
    if len(words) == 0:
        return 0
    
    return urgency_count / len(words) * 100  # 转换为百分比


def calculate_keyword_density(text, keywords):
    """
    计算关键词密度
    """
    if pd.isna(text) or str(text).strip() == '':
        return 0
    
    text_lower = str(text).lower()
    
    # 计算关键词出现次数
    keyword_count = sum(text_lower.count(keyword) for keyword in keywords)
    
    # 归一化
    words = text_lower.split()
    if len(words) == 0:
        return 0
    
    return keyword_count / len(words) * 100


def calculate_emoji_density(text):
    """
    计算emoji密度
    """
    if pd.isna(text) or str(text).strip() == '':
        return 0
    
    text_str = str(text)
    emojis = EMOJI_PATTERN.findall(text_str)
    
    if len(text_str) == 0:
        return 0
    
    return len(emojis) / len(text_str) * 100


def calculate_caps_ratio(text):
    """
    计算大写字母比例
    """
    if pd.isna(text) or str(text).strip() == '':
        return 0
    
    text_str = str(text)
    letters = [c for c in text_str if c.isalpha()]
    
    if len(letters) == 0:
        return 0
    
    caps = [c for c in letters if c.isupper()]
    
    return len(caps) / len(letters) * 100


def calculate_exclamation_ratio(text):
    """
    计算感叹号比例
    """
    if pd.isna(text) or str(text).strip() == '':
        return 0
    
    text_str = str(text)
    
    if len(text_str) == 0:
        return 0
    
    return text_str.count('!') / len(text_str) * 100


def extract_text_features(reddit_df):
    """
    提取文本特征
    """
    
    print("\nExtracting text features...")
    
    # 创建时间窗口
    reddit_df['time_window'] = reddit_df['timestamp'].dt.floor(f'{TIME_WINDOW}min')
    
    # 初始化结果列表
    features_list = []
    
    time_windows = sorted(reddit_df['time_window'].unique())
    
    for i, window_time in enumerate(time_windows):
        if i % 100 == 0:
            print(f"  Processing window {i+1:,}/{len(time_windows):,}...", end='\r')
        
        # 获取该窗口的数据
        window_data = reddit_df[reddit_df['time_window'] == window_time]
        
        # 合并所有文本
        texts = window_data['text'].tolist()
        
        if len(texts) == 0:
            # 空窗口
            features_list.append({
                'timestamp': window_time,
                'sentiment_positive': 0,
                'sentiment_negative': 0,
                'sentiment_neutral': 1,
                'sentiment_compound': 0,
                'urgency_score': 0,
                'bullish_keywords': 0,
                'bearish_keywords': 0,
                'emoji_density': 0,
                'caps_ratio': 0,
                'exclamation_ratio': 0
            })
            continue
        
        # ========== Sentiment ==========
        if VADER_AVAILABLE:
            sentiments = calculate_sentiment_vader(texts)
        else:
            sentiments = calculate_sentiment_simple(texts)
        
        # 平均sentiment
        avg_sentiment = {
            'sentiment_positive': sentiments['positive'].mean(),
            'sentiment_negative': sentiments['negative'].mean(),
            'sentiment_neutral': sentiments['neutral'].mean(),
            'sentiment_compound': sentiments['compound'].mean()
        }
        
        # ========== Urgency ==========
        urgency_scores = [calculate_urgency_score(text) for text in texts]
        avg_urgency = np.mean(urgency_scores) if urgency_scores else 0
        
        # ========== Financial Keywords ==========
        bullish_scores = [calculate_keyword_density(text, BULLISH_KEYWORDS) for text in texts]
        bearish_scores = [calculate_keyword_density(text, BEARISH_KEYWORDS) for text in texts]
        
        avg_bullish = np.mean(bullish_scores) if bullish_scores else 0
        avg_bearish = np.mean(bearish_scores) if bearish_scores else 0
        
        # ========== Emotional Expression ==========
        emoji_scores = [calculate_emoji_density(text) for text in texts]
        caps_scores = [calculate_caps_ratio(text) for text in texts]
        excl_scores = [calculate_exclamation_ratio(text) for text in texts]
        
        avg_emoji = np.mean(emoji_scores) if emoji_scores else 0
        avg_caps = np.mean(caps_scores) if caps_scores else 0
        avg_excl = np.mean(excl_scores) if excl_scores else 0
        
        # 合并所有特征
        window_features = {
            'timestamp': window_time,
            **avg_sentiment,
            'urgency_score': avg_urgency,
            'bullish_keywords': avg_bullish,
            'bearish_keywords': avg_bearish,
            'emoji_density': avg_emoji,
            'caps_ratio': avg_caps,
            'exclamation_ratio': avg_excl
        }
        
        features_list.append(window_features)
    
    print(f"\n  ✓ Extracted text features for {len(features_list):,} windows")
    
    return pd.DataFrame(features_list)


# ============================================================
# 主函数
# ============================================================

def main():
    start_time = datetime.now()
    
    print("\n" + "="*70)
    print("TEXT/NLP FEATURE EXTRACTION")
    print("="*70)
    
    if VADER_AVAILABLE:
        print("✓ Using VADER Sentiment Analyzer")
    else:
        print("⚠️  Using simplified sentiment (install VADER for better results)")
    
    # ========== Step 1: 加载数据 ==========
    print("\nStep 1: Loading Reddit data...")
    
    try:
        reddit_df = pd.read_csv(DATA_FILE)
        reddit_df['timestamp'] = pd.to_datetime(reddit_df['timestamp'])
        
        print(f"  ✓ Loaded {len(reddit_df):,} items")
        print(f"  Date range: {reddit_df['timestamp'].min()} to {reddit_df['timestamp'].max()}")
        
        # 检查text列
        if 'text' not in reddit_df.columns:
            print(f"  ✗ No 'text' column found!")
            return
        
    except FileNotFoundError:
        print(f"  ✗ {DATA_FILE} not found!")
        return
    
    # ========== Step 2: 提取特征 ==========
    print("\nStep 2: Extracting text features...")
    
    text_features = extract_text_features(reddit_df)
    
    # ========== Step 3: 统计 ==========
    print("\nStep 3: Feature statistics...")
    
    print(f"\n  Sentiment statistics:")
    print(f"    Mean positive: {text_features['sentiment_positive'].mean():.3f}")
    print(f"    Mean negative: {text_features['sentiment_negative'].mean():.3f}")
    print(f"    Mean compound: {text_features['sentiment_compound'].mean():.3f}")
    
    print(f"\n  Urgency statistics:")
    print(f"    Mean urgency score: {text_features['urgency_score'].mean():.3f}%")
    print(f"    Max urgency score: {text_features['urgency_score'].max():.3f}%")
    urgent_windows = (text_features['urgency_score'] > 0).sum()
    print(f"    Windows with urgency: {urgent_windows:,}")
    
    print(f"\n  Keyword statistics:")
    print(f"    Mean bullish: {text_features['bullish_keywords'].mean():.3f}%")
    print(f"    Mean bearish: {text_features['bearish_keywords'].mean():.3f}%")
    
    print(f"\n  Expression statistics:")
    print(f"    Mean emoji density: {text_features['emoji_density'].mean():.3f}%")
    print(f"    Mean caps ratio: {text_features['caps_ratio'].mean():.3f}%")
    print(f"    Mean exclamation ratio: {text_features['exclamation_ratio'].mean():.3f}%")
    
    # ========== Step 4: 保存 ==========
    print("\nStep 4: Saving...")
    
    text_features.to_parquet('text_features_5min.parquet', index=False)
    
    print(f"  ✓ Saved: text_features_5min.parquet")
    print(f"  Shape: {text_features.shape}")
    
    # ========== 总结 ==========
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    print("\n" + "="*70)
    print("COMPLETE")
    print("="*70)
    
    print(f"\nTime windows: {len(text_features):,}")
    print(f"Feature columns: {len(text_features.columns)}")
    
    print(f"\nFeatures extracted:")
    print(f"  ✓ Sentiment (4 scores)")
    print(f"  ✓ Urgency score")
    print(f"  ✓ Financial keywords (bullish/bearish)")
    print(f"  ✓ Emotional expression (emoji, caps, exclamation)")
    
    print(f"\nRuntime: {duration:.1f} seconds")
    
    print("\n✓ Feature extraction complete!")
    print("  (Alignment will be done in merge_all_features.py)")


if __name__ == "__main__":
    main()


TEXT/NLP FEATURE EXTRACTION
✓ Using VADER Sentiment Analyzer

Step 1: Loading Reddit data...
  ✓ Loaded 1,606,093 items
  Date range: 2019-07-01 04:35:02 to 2021-06-29 23:58:28

Step 2: Extracting text features...

Extracting text features...
  Processing window 77,201/77,267...
  ✓ Extracted text features for 77,267 windows

Step 3: Feature statistics...

  Sentiment statistics:
    Mean positive: 0.101
    Mean negative: 0.067
    Mean compound: 0.104

  Urgency statistics:
    Mean urgency score: 0.571%
    Max urgency score: 51.923%
    Windows with urgency: 39,282

  Keyword statistics:
    Mean bullish: 2.846%
    Mean bearish: 1.320%

  Expression statistics:
    Mean emoji density: 0.230%
    Mean caps ratio: 13.675%
    Mean exclamation ratio: 0.246%

Step 4: Saving...
  ✓ Saved: text_features_5min.parquet
  Shape: (77267, 11)

COMPLETE

Time windows: 77,267
Feature columns: 11

Features extracted:
  ✓ Sentiment (4 scores)
  ✓ Urgency score
  ✓ Financial keywords (bullish/bea